In [1]:
# Cell 1 — Environment probe
import subprocess, sys, os, shutil

print("=" * 50)
print("GPU CHECK")
print("=" * 50)
subprocess.run(["nvidia-smi"], check=False)

print("\n" + "=" * 50)
print("PYTHON & LIBS")
print("=" * 50)
print(f"Python: {sys.version.split()[0]}")

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB" if torch.cuda.is_available() else "")

# Install what's not preinstalled
subprocess.run([sys.executable, "-m", "pip", "install", "-q", 
                "ultralytics==8.3.40", "wandb", "mlflow", "roboflow"], check=True)

import ultralytics, wandb, mlflow, roboflow
print(f"\nultralytics: {ultralytics.__version__}")
print(f"wandb: {wandb.__version__}")
print(f"mlflow: {mlflow.__version__}")
print(f"roboflow: {roboflow.__version__}")

print("\n" + "=" * 50)
print("DISK")
print("=" * 50)
total, used, free = shutil.disk_usage("/kaggle/working")
print(f"/kaggle/working: {free / 1e9:.1f} GB free of {total / 1e9:.1f} GB")

print("\n✅ Env probe passed" if torch.cuda.is_available() else "\n❌ NO GPU — fix accelerator setting")

GPU CHECK
Wed May 13 15:21:07 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   52C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-------------------------------------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2026.2.0 which is incompatible.


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.

ultralytics: 8.3.40
wandb: 0.25.0
mlflow: 3.12.0
roboflow: 1.3.8

DISK
/kaggle/working: 20.9 GB free of 21.0 GB

✅ Env probe passed


In [2]:
# Cell 2 — Secrets probe
from kaggle_secrets import UserSecretsClient
import os

user_secrets = UserSecretsClient()

results = {}
for secret_name in ["ROBOFLOW_API_KEY", "WANDB_API_KEY", "HF_TOKEN"]:
    try:
        value = user_secrets.get_secret(secret_name)
        os.environ[secret_name] = value
        # Show only length + last 4 chars (never log full secrets)
        results[secret_name] = f"✅ loaded (len={len(value)}, ends ...{value[-4:]})"
    except Exception as e:
        results[secret_name] = f"❌ FAILED: {type(e).__name__}: {str(e)[:80]}"

print("=" * 50)
print("SECRETS")
print("=" * 50)
for k, v in results.items():
    print(f"{k}: {v}")

all_ok = all("✅" in v for v in results.values())
print(f"\n{'✅ All secrets loaded' if all_ok else '❌ Fix missing secrets before continuing'}")

SECRETS
ROBOFLOW_API_KEY: ✅ loaded (len=20, ends ...3riz)
WANDB_API_KEY: ✅ loaded (len=86, ends ...N62S)
HF_TOKEN: ✅ loaded (len=37, ends ...uvYk)

✅ All secrets loaded


In [3]:
# Cell 3 — Roboflow dataset download: PPE-Combined (forked, ~58k images, 13 classes)
import os
from pathlib import Path
import yaml
from roboflow import Roboflow

rf = Roboflow(api_key=os.environ["ROBOFLOW_API_KEY"])
os.chdir("/kaggle/working")

WORKSPACE = "mazz-maxx"
PROJECT = "ppe-combined-9bprl-mmcaf"
VERSION_NUM = 1

project = rf.workspace(WORKSPACE).project(PROJECT)
version = project.version(VERSION_NUM)
dataset = version.download("yolov8")

print("\n" + "=" * 50); print("DATASET PROBE"); print("=" * 50)
dataset_path = Path(dataset.location)
print(f"Dataset path: {dataset_path}")

yaml_path = dataset_path / "data.yaml"
assert yaml_path.exists(), f"❌ data.yaml not found"
with open(yaml_path) as f:
    data_cfg = yaml.safe_load(f)

print(f"\ndata.yaml:")
for k, v in data_cfg.items():
    if k != "roboflow":
        print(f"  {k}: {v}")

for split in ["train", "valid", "test"]:
    split_dir = dataset_path / split / "images"
    if split_dir.exists():
        n = len(list(split_dir.glob("*.*")))
        print(f"  {split}: {n} images")

n_classes = data_cfg.get("nc", len(data_cfg.get("names", [])))
class_names = data_cfg.get("names", [])
print(f"\nClasses ({n_classes}): {class_names}")

DATASET_YAML = str(yaml_path)
print(f"\n✅ DATASET_YAML = {DATASET_YAML}")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to PPE-Combined-1 in yolov8:: 100%|██████████| 115813/115813 [00:27<00:00, 4241.32it/s]



DATASET PROBE
Dataset path: /kaggle/working/PPE-Combined-1

data.yaml:
  names: ['Fall-Detected', 'Gloves', 'Goggles', 'Hardhat', 'Mask', 'NO-Gloves', 'NO-Goggles', 'NO-Hardhat', 'NO-Mask', 'NO-Safety Vest', 'No_Harness', 'Person', 'Safety Vest']
  nc: 13
  test: ../test/images
  train: ../train/images
  val: ../valid/images
  train: 41922 images
  valid: 10834 images
  test: 5148 images

Classes (13): ['Fall-Detected', 'Gloves', 'Goggles', 'Hardhat', 'Mask', 'NO-Gloves', 'NO-Goggles', 'NO-Hardhat', 'NO-Mask', 'NO-Safety Vest', 'No_Harness', 'Person', 'Safety Vest']

✅ DATASET_YAML = /kaggle/working/PPE-Combined-1/data.yaml


In [4]:
# Cell 6 (RESUME VERSION) — copy prior artifacts, continue W&B + YOLO resume
import subprocess, sys, os, shutil

subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "ray"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "onnxscript"], check=True)

# Copy previous training artifacts from input dataset to /kaggle/working
# (need writable path for ultralytics resume; /kaggle/input/ is read-only)
SRC = "/kaggle/input/datasets/ayushgupta07xx/safetyvision-v1-checkpoints/safetyvision"
DST = "/kaggle/working/safetyvision"
print(f"Copying {SRC} → {DST}...")
if not os.path.exists(DST):
    shutil.copytree(SRC, DST)
print(f"✅ Previous run artifacts copied")

LAST_PT = f"{DST}/yolov8n-ppe-v1/weights/last.pt"
assert os.path.exists(LAST_PT), f"❌ last.pt not found at {LAST_PT}"
print(f"✅ last.pt found ({os.path.getsize(LAST_PT) / 1e6:.1f} MB)")

# W&B — continue the crashed run
import wandb
wandb.login(key=os.environ["WANDB_API_KEY"], relogin=True)
os.environ["WANDB_PROJECT"] = "safetyvision"

PREVIOUS_RUN_ID = "9nctv2ai"
run = wandb.init(
    project="safetyvision",
    id=PREVIOUS_RUN_ID,
    resume="must",
    settings=wandb.Settings(init_timeout=300),
)
print(f"✅ W&B run resumed: {run.url}")

# YOLO resume
from ultralytics import YOLO
model = YOLO(LAST_PT)
results = model.train(
    data=DATASET_YAML,
    resume=True,
    project="/kaggle/working/safetyvision",
    name="yolov8n-ppe-v1",
    exist_ok=True,
)

print(f"\n✅ Resume training complete. W&B: {wandb.run.url}")
wandb.finish()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 714.8/714.8 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 13.8 MB/s eta 0:00:00
Copying /kaggle/input/datasets/ayushgupta07xx/safetyvision-v1-checkpoints/safetyvision → /kaggle/working/safetyvision...


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: agcr7jw (agcr7jw-vellore-institute-of-technology) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING Changes to your `wandb` environment variables will be ignored because your `wandb` session has already started. For more information on how to modify your settings with `wandb.init()` arguments, please refer to https://wandb.me/wandb-init.


✅ Previous run artifacts copied
✅ last.pt found (12.3 MB)


wandb: Tracking run with wandb version 0.25.0
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260513_152343-9nctv2ai
wandb: Run `wandb offline` to turn off syncing.
wandb: Resuming run yolov8n-ppe-v1
wandb: ⭐️ View project at https://wandb.ai/agcr7jw-vellore-institute-of-technology/safetyvision
wandb: 🚀 View run at https://wandb.ai/agcr7jw-vellore-institute-of-technology/safetyvision/runs/9nctv2ai


✅ W&B run resumed: https://wandb.ai/agcr7jw-vellore-institute-of-technology/safetyvision/runs/9nctv2ai
New https://pypi.org/project/ultralytics/8.4.50 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.40 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: task=detect, mode=train, model=/kaggle/working/safetyvision/yolov8n-ppe-v1/weights/last.pt, data=/kaggle/working/PPE-Combined-1/data.yaml, epochs=100, time=None, patience=20, batch=32, imgsz=640, save=True, save_period=10, cache=False, device=0, workers=8, project=/kaggle/working/safetyvision, name=yolov8n-ppe-v1, exist_ok=True, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=/kaggle/working/safetyvision/yolov8n-ppe-v1/weights/last.pt, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hy

...

                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 170/170 [01:12<00:00,  2.35it/s]


                   all      10834      25308       0.63      0.716      0.693      0.431
         Fall-Detected        899        899      0.756      0.852      0.866      0.578
                Gloves        846       1684      0.782      0.685      0.783      0.389
               Goggles        859        964      0.784      0.864      0.892      0.525
               Hardhat       4070      10259      0.799        0.9       0.89      0.522
                  Mask        317        594      0.485      0.875      0.532      0.379
             NO-Gloves        808       1813      0.704      0.682      0.722      0.344
            NO-Goggles        999       1372      0.771       0.63       0.73      0.395
            NO-Hardhat       1147       2707      0.596      0.793      0.706      0.461
               NO-Mask        327        505      0.574      0.788      0.611      0.446
        NO-Safety Vest        409        698      0.398      0.494      0.358      0.208
            No_Harnes

wandb: updating run metadata



✅ Resume training complete. W&B: https://wandb.ai/agcr7jw-vellore-institute-of-technology/safetyvision/runs/9nctv2ai


wandb: 🚀 View run yolov8n-ppe-v1 at: https://wandb.ai/agcr7jw-vellore-institute-of-technology/safetyvision/runs/9nctv2ai
wandb: ⭐️ View project at: https://wandb.ai/agcr7jw-vellore-institute-of-technology/safetyvision
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./wandb/run-20260513_152343-9nctv2ai/logs


In [5]:
# Cell 7 — Final eval (val + test) + ONNX export
from pathlib import Path
from ultralytics import YOLO

best_pt = Path("/kaggle/working/safetyvision/yolov8n-ppe-v1/weights/best.pt")
assert best_pt.exists(), f"best.pt missing at {best_pt}"

m = YOLO(str(best_pt))

print("=" * 50); print("VAL SPLIT"); print("=" * 50)
val_metrics = m.val(data=DATASET_YAML, split="val", imgsz=640, device=0)
print(f"mAP50: {val_metrics.box.map50:.4f}")
print(f"mAP50-95: {val_metrics.box.map:.4f}")

print("\n" + "=" * 50); print("TEST SPLIT (held out)"); print("=" * 50)
test_metrics = m.val(data=DATASET_YAML, split="test", imgsz=640, device=0)
print(f"Test mAP50: {test_metrics.box.map50:.4f}")
print(f"Test mAP50-95: {test_metrics.box.map:.4f}")

print("\n" + "=" * 50); print("ONNX EXPORT"); print("=" * 50)
onnx_path = m.export(format="onnx", imgsz=640, opset=18, simplify=True)
print(f"✅ ONNX: {onnx_path}")

import onnxruntime as ort
sess = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
print(f"✅ ONNX loadable. Output spec: {[o.shape for o in sess.get_outputs()]}")

VAL SPLIT
Ultralytics 8.3.40 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 168 layers, 3,008,183 parameters, 0 gradients, 8.1 GFLOPs


val: Scanning /kaggle/working/PPE-Combined-1/valid/labels.cache... 10834 images, 920 backgrounds, 0 corrupt: 100%|██████████| 10834/10834 [00:00<?, ?it/s]

WARNING ⚠️ Box and segment counts should be equal, but got len(segments) = 26, len(boxes) = 25308. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.



                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 678/678 [01:26<00:00,  7.85it/s]


                   all      10834      25308      0.629      0.716      0.693      0.431
         Fall-Detected        899        899       0.75      0.851      0.865      0.578
                Gloves        846       1684      0.782      0.686      0.783      0.389
               Goggles        859        964      0.785      0.864      0.892      0.524
               Hardhat       4070      10259        0.8        0.9       0.89      0.522
                  Mask        317        594      0.484      0.875      0.532      0.379
             NO-Gloves        808       1813      0.703      0.683      0.721      0.344
            NO-Goggles        999       1372      0.771      0.629      0.729      0.396
            NO-Hardhat       1147       2707      0.596      0.793      0.706      0.461
               NO-Mask        327        505      0.573      0.788      0.611      0.446
        NO-Safety Vest        409        698      0.398      0.494      0.359      0.208
            No_Harnes

...

val: New cache created: /kaggle/working/PPE-Combined-1/test/labels.cache


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 322/322 [00:40<00:00,  7.88it/s]


                   all       5148      11303      0.607      0.711      0.701      0.441
         Fall-Detected        450        450      0.775      0.793      0.838      0.555
                Gloves        276        569      0.765      0.715      0.802      0.416
               Goggles        426        470      0.798       0.86      0.908      0.528
               Hardhat       2050       5100      0.813      0.883      0.891      0.518
                  Mask        176        434      0.568      0.733      0.651      0.394
             NO-Gloves        306        676       0.74      0.754      0.802        0.4
            NO-Goggles        428        558       0.78      0.701      0.799      0.458
            NO-Hardhat        471       1122      0.621      0.821      0.775      0.521
               NO-Mask        158        218      0.502      0.763      0.573      0.378
        NO-Safety Vest        175        342      0.469      0.491      0.395      0.217
            No_Harnes

W0513 18:14:13.430000 23 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, rois, spatial_scale: 'float', pooled_height: 'int', pooled_width: 'int', sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0513 18:14:13.432000 23 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'rois' from (input, rois, spatial_scale: 'float', pooled_height: 'int', pooled_width: 'int', sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0513 18:14:13.434000 23 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.
W0513 18:14:13.436000 23 torch/onnx/_internal/exporter/_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0). Treating as an Input.


ONNX: slimming with onnxslim 0.1.93...
ONNX: export success ✅ 19.5s, saved as '/kaggle/working/safetyvision/yolov8n-ppe-v1/weights/best.onnx' (11.6 MB)

Export complete (21.1s)
Results saved to /kaggle/working/safetyvision/yolov8n-ppe-v1/weights
Predict:         yolo predict task=detect model=/kaggle/working/safetyvision/yolov8n-ppe-v1/weights/best.onnx imgsz=640  
Validate:        yolo val task=detect model=/kaggle/working/safetyvision/yolov8n-ppe-v1/weights/best.onnx imgsz=640 data=/kaggle/working/PPE-Combined-1/data.yaml  
Visualize:       https://netron.app
✅ ONNX: /kaggle/working/safetyvision/yolov8n-ppe-v1/weights/best.onnx
✅ ONNX loadable. Output spec: [[1, 17, 8400]]


In [6]:
# Cell 8 — Output summary (Save Version auto-zips /kaggle/working)
from pathlib import Path

print("=" * 50); print("OUTPUTS"); print("=" * 50)
working = Path("/kaggle/working")
artifacts = [
    "safetyvision/yolov8n-ppe-v1/weights/best.pt",
    "safetyvision/yolov8n-ppe-v1/weights/last.pt",
    "safetyvision/yolov8n-ppe-v1/weights/best.onnx",
    "safetyvision/yolov8n-ppe-v1/results.csv",
    "safetyvision/yolov8n-ppe-v1/confusion_matrix.png",
    "safetyvision/yolov8n-ppe-v1/F1_curve.png",
    "safetyvision/yolov8n-ppe-v1/results.png",
    "runs/mlflow",
    "wandb",
]
for a in artifacts:
    p = working / a
    if p.exists():
        if p.is_file():
            print(f"  ✅ {a} ({p.stat().st_size / 1e6:.1f} MB)")
        else:
            print(f"  ✅ {a}/ ({sum(1 for _ in p.rglob('*'))} items)")
    else:
        print(f"  ❌ MISSING: {a}")

print("\nAll artifacts in /kaggle/working — auto-attached to Save Version output.")

OUTPUTS
  ✅ safetyvision/yolov8n-ppe-v1/weights/best.pt (6.2 MB)
  ✅ safetyvision/yolov8n-ppe-v1/weights/last.pt (6.2 MB)
  ✅ safetyvision/yolov8n-ppe-v1/weights/best.onnx (12.2 MB)
  ✅ safetyvision/yolov8n-ppe-v1/results.csv (0.0 MB)
  ✅ safetyvision/yolov8n-ppe-v1/confusion_matrix.png (0.3 MB)
  ✅ safetyvision/yolov8n-ppe-v1/F1_curve.png (0.5 MB)
  ✅ safetyvision/yolov8n-ppe-v1/results.png (0.2 MB)
  ✅ runs/mlflow/ (174 items)
  ✅ wandb/ (17 items)

All artifacts in /kaggle/working — auto-attached to Save Version output.
